# 02 — Extract Toronto Census-Tract Data from the StatCan Profile Bulk File

**Placement in the pipeline:** this is a one-time, non-reproducible-on-this-machine extraction step
(the 2.6GB bulk source file is not checked into the repo and is no longer on disk — see below). It
sits right after notebook 01 (election normalization) because notebook 03
(`03_build_census_tract_profile.ipynb`) depends directly on this notebook's two output files as its
**primary** source — not a cached workaround, as an earlier version of notebook 03 had to use before
this extraction existed.

Source: `data/census/98-401-X2021007_eng_CSV/98-401-X2021007_English_CSV_data.csv` — the 2021 Census Profile
in long format for every geography in Canada (16.5M rows, 2.6 GB).

Outputs (in `data/census/`):

- `toronto-ct.csv` — the **full** extraction: every column and every characteristic for the City of Toronto's census tracts.
- `toronto-ct-small.csv` — only `GEO_NAME, CHARACTERISTIC_ID, CHARACTERISTIC_NAME, CHARACTERISTIC_CODE, C1_COUNT_TOTAL, C10_RATE_TOTAL`.

**Strategy:** use StatCan's geography index (`Geo_starting_row.CSV`) to find the line range of the Toronto CMA's census tracts, stream only that range,
and keep rows whose CT id is one of the 622 City of Toronto CTs (the CMA also includes Mississauga, Brampton, etc.).

`CHARACTERISTIC_CODE` is not in the StatCan file; it is a short label from the project's curated `toronto-ada.csv`, joined on `CHARACTERISTIC_ID`.
Characteristics not in that curated list have an empty code in the small file.

**Reproducibility note:** the bulk source file has since been deleted from this machine (it's gitignored
and was never meant to be checked in). This notebook's own outputs (`toronto-ct.csv`, `toronto-ct-small.csv`)
are the durable, locally-cached artifacts that notebook 03 and later notebooks actually read; re-running
this notebook from scratch requires re-downloading the StatCan bulk Census Profile product (98-401-X2021007)
for all of Canada.

In [1]:
import csv, itertools
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

In [2]:
ROOT = Path("../..")

CENSUS_DIR = ROOT / "data/census"
BULK_DIR = CENSUS_DIR / "98-401-X2021007_eng_CSV"
BULK_DATA_CSV = BULK_DIR / "98-401-X2021007_English_CSV_data.csv"
GEO_INDEX_CSV = BULK_DIR / "98-401-X2021007_Geo_starting_row.CSV"

CODE_LOOKUP_CSV = ROOT / "data/clustering_neighbourhoods/census/toronto-ada.csv"  # CHARACTERISTIC_ID -> CHARACTERISTIC_CODE
CT_PROFILE_CSV = ROOT / "analysis/toronto_election_turnout/data/archive/census/processed/ct/statcan_2021_ct_profile.csv"  # 622 City of Toronto CTs

OUTPUT_FULL = CENSUS_DIR / "toronto-ct.csv"
OUTPUT_SMALL = CENSUS_DIR / "toronto-ct-small.csv"
SMALL_COLUMNS = ["GEO_NAME", "CHARACTERISTIC_ID", "CHARACTERISTIC_NAME", "CHARACTERISTIC_CODE", "C1_COUNT_TOTAL", "C10_RATE_TOTAL"]

BULK_ENCODING = "latin-1"
TORONTO_CMA_GEO_CODE = "2021S0503535"
TORONTO_CT_DGUID_PREFIX = "2021S0507535"  # CTs in the Toronto CMA (535)

## 1. Target census tracts and characteristic codes

In [3]:
target_ct_geo_ids = set(pd.read_csv(CT_PROFILE_CSV, dtype=str, usecols=["geo_id"])["geo_id"])

code_lookup = (
    pd.read_csv(CODE_LOOKUP_CSV, dtype=str)
    .drop_duplicates("CHARACTERISTIC_ID")
    .set_index("CHARACTERISTIC_ID")["CHARACTERISTIC_CODE"]
)
print(f"Target CTs: {len(target_ct_geo_ids)}; coded characteristics: {len(code_lookup)}")

Target CTs: 622; coded characteristics: 184


## 2. Locate the Toronto CMA census-tract line range

In [4]:
with open(GEO_INDEX_CSV, encoding=BULK_ENCODING, newline="") as f:
    geo_rows = list(csv.DictReader(f))

toronto_idx = next(i for i, r in enumerate(geo_rows) if r["Geo Code"] == TORONTO_CMA_GEO_CODE and r["Geo Name"] == "Toronto")
j = toronto_idx + 1
while j < len(geo_rows) and geo_rows[j]["Geo Code"].startswith(TORONTO_CT_DGUID_PREFIX):
    j += 1

first_line = int(geo_rows[toronto_idx + 1]["Line Number"])
last_line = int(geo_rows[j]["Line Number"]) - 1
print(f"CMA census tracts: {j - toronto_idx - 1}; lines {first_line:,}-{last_line:,}")

CMA census tracts: 1227; lines 10,818,674-14,046,910


## 3. Stream the slice and keep City of Toronto CTs (all columns, all characteristics)

In [5]:
n_rows = last_line - first_line + 1
kept = []
with open(BULK_DATA_CSV, encoding=BULK_ENCODING, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    geo_col = header.index("GEO_NAME")
    for _ in itertools.islice(reader, first_line - 2):
        pass
    for row in tqdm(itertools.islice(reader, n_rows), total=n_rows):
        if row[geo_col] in target_ct_geo_ids:
            kept.append(row)
print(f"Kept {len(kept):,} rows")

  0%|          | 0/3228237 [00:00<?, ?it/s]

Kept 1,636,482 rows


## 4. Assemble and save

The raw file has repeated `SYMBOL` column names; pandas would rename them `SYMBOL.1`, … on read, so the full output is written with the original header via the `csv` module's rows through a DataFrame with positional columns.

In [6]:
full = pd.DataFrame(kept, columns=header)  # duplicate SYMBOL names preserved
full["_id"] = full["CHARACTERISTIC_ID"].astype(int)
full = full.sort_values(["GEO_NAME", "_id"], kind="stable").drop(columns="_id").reset_index(drop=True)
full.to_csv(OUTPUT_FULL, index=False)

small = full.loc[:, ~full.columns.duplicated()].copy()
small["CHARACTERISTIC_CODE"] = small["CHARACTERISTIC_ID"].map(code_lookup).fillna("")
small = small[SMALL_COLUMNS]
small.to_csv(OUTPUT_SMALL, index=False)
print(f"Wrote {len(full):,} rows x {full.shape[1]} cols to {OUTPUT_FULL}")
print(f"Wrote {len(small):,} rows x {small.shape[1]} cols to {OUTPUT_SMALL}")
small.head()

Wrote 1,636,482 rows x 23 cols to ../../data/census/toronto-ct.csv
Wrote 1,636,482 rows x 6 cols to ../../data/census/toronto-ct-small.csv


,GEO_NAME,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_CODE,C1_COUNT_TOTAL,C10_RATE_TOTAL
0,5350001.00,1,"Population, 2021",pop_2021,599,
1,5350001.00,2,"Population, 2016",pop_2016,595,
2,5350001.00,3,"Population percentage change, 2016 to 2021",pop_pct_change,0.7,0.7
3,5350001.00,4,Total private dwellings,,253,
4,5350001.00,5,Private dwellings occupied by usual residents,,235,


## 5. Sanity checks

In [7]:
found = set(full["GEO_NAME"])
dupes = full.duplicated(subset=["GEO_NAME", "CHARACTERISTIC_ID"]).sum()
per_ct = full.groupby("GEO_NAME").size()
print(f"CTs found: {len(found)} / {len(target_ct_geo_ids)}; duplicates: {dupes}; rows per CT: {per_ct.min()}-{per_ct.max()}")
assert found == target_ct_geo_ids and dupes == 0 and per_ct.nunique() == 1

CTs found: 622 / 622; duplicates: 0; rows per CT: 2631-2631
